# claim&nbsp;1 &mdash; post-occlusion coverage

After an occlusion, do the **memory oracle** and **mask oracle** recover the target better than the plain **sam** baseline &mdash; and does the gap widen the longer the occlusion lasts?

This notebook draws the figures from `data/claim_1/results.pkl` (produced by `claim_1.py`). It is pure numpy/matplotlib &mdash; no tracker imports &mdash; so it redraws instantly. Each figure is shown inline and saved next to the pkl.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

METRIC_IOU = 0.5     # coverage threshold to plot

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 12, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "--",
    "legend.frameon": False, "lines.linewidth": 2.2, "lines.markersize": 7,
})
COLORS = {"mask": "#d62728", "memory": "#1f77b4", "sam": "#8d99ae"}   # oracles in colour, baseline in grey

# locate the repo root (the folder that contains conf/) whether the notebook runs from notebooks/ or the root
root = Path.cwd()
while not (root / "conf").exists() and root != root.parent:
    root = root.parent
out_dir = root / "data" / "claim_1"

state = pickle.load(open(out_dir / "results.pkl", "rb"))
n_bins = state["n_bins"]
clips = state["clips"]

# results.pkl holds one record per trajectory (raw per-frame IoUs at a fixed 0.2 commit gate); collapse to coverage
def coverage(ious, t=METRIC_IOU):
    return float(np.mean(np.asarray(ious) >= t)) if len(ious) else np.nan

occ_count = np.array([c["occ_count"] for c in clips])
sam_cov = np.array([coverage(c["sam"]) for c in clips])
memory_cov = np.array([coverage(c["memory"]) for c in clips])
mask_cov = np.array([coverage(c["mask"]) for c in clips])
ylabel = f"post-occlusion coverage (IoU $\\geq$ {METRIC_IOU})"
print(f"{len(clips)} trajectories, METRIC_IOU={METRIC_IOU}  "
      f"means: sam={np.nanmean(sam_cov):.3f} memory={np.nanmean(memory_cov):.3f} mask={np.nanmean(mask_cov):.3f}")

In [ ]:
edges = np.quantile(occ_count, np.linspace(0, 1, n_bins + 1))
edges[-1] += 1e-6                                              # include the max in the last bin
bin_of = np.clip(np.digitize(occ_count, edges) - 1, 0, n_bins - 1)
present = [b for b in range(n_bins) if (bin_of == b).any()]
centers = [float(occ_count[bin_of == b].mean()) for b in present]

def binned_mean(values):
    values = np.asarray(values, dtype=float)
    return np.array([np.nanmean(values[bin_of == b]) for b in present])

fig, ax = plt.subplots(figsize=(7.5, 5))
for key, label, values, style in [("mask", "mask oracle", mask_cov, "s-"),
                                   ("memory", "memory oracle", memory_cov, "o-"),
                                   ("sam", "sam baseline", sam_cov, "^--")]:
    ax.plot(centers, binned_mean(values), style, color=COLORS[key], label=label)

ax.set_xlabel("number of occluded frames")
ax.set_ylabel(ylabel)
ax.set_ylim(0, 1.02)
ax.set_title(f"post-occlusion coverage vs. occlusion  (n = {len(clips)})")
ax.legend(loc="lower left")
fig.savefig(out_dir / "claim_1_occlusion.png")
plt.show()